# 01 — Data and Pipeline Check

This notebook validates the project configuration, frozen participant-level NACC split, 2.5D stack format, and PyTorch dataset pipeline.

It uses reusable functions from the `src/` directory rather than redefining the project pipeline.

**Controlled NACC and ADNI data are not included in the repository.**


## 1. Project Setup

The following cell locates the repository root and makes the reusable `src/` modules available to Python.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MRI_XAI_Robustness"
)

SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Project root:', PROJECT_ROOT)
print('Source directory:', SRC_DIR)


## 2. Reproducibility and Device

The project uses a fixed random seed of 42 for reproducible data splitting and development experiments.

The value 42 has no scientific significance; it is simply a fixed pseudorandom starting state.


In [ ]:
from utils import DEFAULT_SEED, set_random_seed, get_device

set_random_seed(DEFAULT_SEED)
device = get_device()

print('Random seed:', DEFAULT_SEED)
print('Device:', device)

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 3. Controlled Data

NACC and ADNI imaging data and participant-level manifests are not distributed through this repository.

Authorized users must obtain the datasets directly from the respective data providers.

Expected frozen NACC split:

| Partition | Normal | AD Dementia | Total |
| --- | ---: | ---: | ---: |
| Training | 2,394 | 365 | 2,759 |
| Validation | 513 | 78 | 591 |
| Test | 513 | 79 | 592 |
| **Total** | **3,420** | **522** | **3,942** |


## 4. Frozen NACC Split Check

When the authorized manifests are available locally, this section checks the expected participant counts and verifies that there is no participant overlap between training, validation, and test partitions.


In [ ]:
MANIFEST_DIR = PROJECT_ROOT / 'data' / 'manifests'

train_path = MANIFEST_DIR / 'nacc_train.csv'
validation_path = MANIFEST_DIR / 'nacc_validation.csv'
test_path = MANIFEST_DIR / 'nacc_test.csv'

required_manifests = [
    train_path,
    validation_path,
    test_path,
]

if all(path.exists() for path in required_manifests):

    train_df = pd.read_csv(train_path)
    validation_df = pd.read_csv(validation_path)
    test_df = pd.read_csv(test_path)

    print('Train:', len(train_df))
    print('Validation:', len(validation_df))
    print('Test:', len(test_df))

    assert len(train_df) == 2759
    assert len(validation_df) == 591
    assert len(test_df) == 592

    train_ids = set(train_df['NACCID'])
    validation_ids = set(validation_df['NACCID'])
    test_ids = set(test_df['NACCID'])

    assert train_ids.isdisjoint(validation_ids)
    assert train_ids.isdisjoint(test_ids)
    assert validation_ids.isdisjoint(test_ids)

    print(
        'PASS: Frozen participant-level split is '
        'present and mutually exclusive.'
    )

else:

    print(
        'Controlled NACC manifests are not available '
        'in this local environment.'
    )


## 5. 2.5D MRI Input Design

Following preprocessing, each participant contributes five overlapping three-slice stacks from the standardized hippocampal/medial-temporal coronal window.

The selected coronal slices are 107–117:

```text
Stack 1: 107, 108, 109
Stack 2: 109, 110, 111
Stack 3: 111, 112, 113
Stack 4: 113, 114, 115
Stack 5: 115, 116, 117
```

Each saved stack has shape `182 × 182 × 3`.

The dataset module converts the stored H × W × 3 representation to channel-first format and dynamically resizes it to `3 × 224 × 224` using bicubic interpolation.


## 6. PyTorch Dataset Check

The reusable `NACC25DDataset` loads the normalized 2.5D stacks and prepares them for model input.

The following check runs only when the local development manifest and derived stack directory are available.


In [ ]:
from dataset import NACC25DDataset

development_manifest_path = (
    MANIFEST_DIR
    / 'nacc_pipeline_development_50.csv'
)

development_stack_root = (
    PROJECT_ROOT
    / 'data'
    / 'pipeline_development_50'
    / '25d_stacks_normalized'
)

if (
    development_manifest_path.exists()
    and development_stack_root.exists()
):

    development_df = pd.read_csv(
        development_manifest_path
    )

    dataset = NACC25DDataset(
        manifest_df=development_df,
        stack_root=development_stack_root,
        output_size=(224, 224),
    )

    sample = dataset[0]

    print('Participants:', len(development_df))
    print('Dataset samples:', len(dataset))
    print('Image shape:', sample['image'].shape)
    print('Image dtype:', sample['image'].dtype)
    print('Label:', sample['label'])
    print('Participant:', sample['participant_id'])
    print('Stack:', sample['stack_number'])

    assert sample['image'].shape == (3, 224, 224)
    assert sample['image'].dtype == torch.float32

    print(
        'PASS: 2.5D dataset pipeline is functioning.'
    )

else:

    print(
        'Local development stacks are not available. '
        'Dataset execution check skipped.'
    )


## 7. Summary

This notebook provides a lightweight reproducibility check for the data-facing portion of the project.

The reusable implementation is located in:

- `src/preprocessing.py`
- `src/dataset.py`
- `src/split_data.py`
- `src/utils.py`

Controlled MRI data and participant-level metadata remain outside version control.
